# Read flight data where it lives
Ask the same SQL question of a Lakehouse file, a Delta table, and Azure Blob Storage.
<br>Then reach the Blob file through a OneLake shortcut.
**Before you begin:** Use Pure Python and attach `lh_pond` as the default Lakehouse.
<br>Run `0. Setup` with `DATA_SCOPE = "one_year"`, including its Delta table step.
<br>Section 4 also needs a `blob_flights` shortcut to your own Blob container.
<br>Run the connection cell first. Each source example then defines its own path.
<br>The public Blob, shortcut, private Blob, and Excel examples are optional.


## One question
**How many flights departed Barcelona in January 2025?**
<br>Keep the `SELECT` and filter. Change the source in `FROM`.
Run this cell to open a DuckDB connection.
<br>The examples use the same January data, so their counts should match.


In [ ]:
import duckdb
con = duckdb.connect()


## 1. A file in the Lakehouse
Use the familiar Lakehouse **Files** path.
<br>`read_parquet` reads January’s file; `LEBL` selects Barcelona departures.


In [ ]:
MOUNT = "/lakehouse/default/Files/flights_eu/one_year/flight_list_202501.parquet"

con.sql(f"""
    SELECT count(*) AS flights_from_barcelona
    FROM read_parquet('{MOUNT}')
    WHERE adep = 'LEBL'
""").show()


## 2. A Delta table in the Lakehouse
Setup created **`dbo.flights_202501`** from the January file.
<br>`delta_scan` uses the transaction log to identify the table’s current data files.
**INSTALL** makes the extension available. **LOAD** enables it for this connection.
<br>This query reads Delta storage directly; it does not use the SQL endpoint.


In [ ]:
TABLE = "/lakehouse/default/Tables/dbo/flights_202501"

con.execute("INSTALL delta")
con.execute("LOAD delta")

con.sql(f"""
    SELECT count(*) AS flights_from_barcelona
    FROM delta_scan('{TABLE}')
    WHERE adep = 'LEBL'
""").show()


## 3. A public file in Azure Blob Storage
The **`httpfs` extension** reads Parquet over HTTPS.
<br>Upload the January 2025 flight Parquet file downloaded by `0. Setup` to your own public Blob container.
<br>Set `PUBLIC_BLOB` to its HTTPS URL, then set `RUN_PUBLIC_BLOB = True`. Leave it `False` to skip.
<br>Use the same dataset to preserve the expected columns and Barcelona flight count.


In [ ]:
RUN_PUBLIC_BLOB = False
PUBLIC_BLOB = "https://YOUR-STORAGE-ACCOUNT.blob.core.windows.net/YOUR-CONTAINER/flight_list_202501.parquet"

if RUN_PUBLIC_BLOB:
    if "YOUR-" in PUBLIC_BLOB:
        raise ValueError("Set PUBLIC_BLOB to your own public Parquet URL before enabling this example.")

    con.execute("INSTALL httpfs")
    con.execute("LOAD httpfs")

    con.sql(f"""
        SELECT count(*) AS flights_from_barcelona
        FROM read_parquet('{PUBLIC_BLOB}')
        WHERE adep = 'LEBL'
    """).show()
else:
    print("Public Blob example skipped. Set your URL, then RUN_PUBLIC_BLOB = True.")


## 4. The Blob file through a OneLake shortcut
**Same Blob file, now reached through a Lakehouse path.**
<br>The shortcut exposes it under **Files → blob_flights**.
Create the shortcut to your own Blob container containing `flight_list_202501.parquet` before running this cell.
<br>Set `RUN_SHORTCUT = True` after creating it. Leave it `False` to skip.


In [ ]:
RUN_SHORTCUT = False

if RUN_SHORTCUT:
    SHORTCUT = "/lakehouse/default/Files/blob_flights/flight_list_202501.parquet"

    con.sql(f"""
        SELECT count(*) AS flights_from_barcelona
        FROM read_parquet('{SHORTCUT}')
        WHERE adep = 'LEBL'
    """).show()
else:
    print("Shortcut example skipped. Create blob_flights, then set RUN_SHORTCUT = True.")


## Optional: a private Azure Blob via Key Vault
**Key Vault supplies the credential; DuckDB reads the file.**
<br>The `azure` extension uses a SAS token to access private Blob Storage.
Set `RUN_PRIVATE_BLOB = True` only after configuring your vault, secret, and Blob address.
<br>You need permission to retrieve the secret and a valid SAS for the file.
<br>Leave it `False` to skip. Keep the SAS out of notebook output.


In [ ]:
RUN_PRIVATE_BLOB = False

if RUN_PRIVATE_BLOB:
    from notebookutils import credentials

    PRIVATE_BLOB = "azure://YOUR-CONTAINER/flight_list_202501.parquet"

    con.execute("INSTALL azure")
    con.execute("LOAD azure")

    # Retrieve the SAS token from Key Vault.
    sas_token = credentials.getSecret(
        "https://YOUR-KEY-VAULT.vault.azure.net/",
        "storage-sas-token",
    ).strip().lstrip("?")

    # Give DuckDB the credential for this storage account.
    con.execute(f"""
        CREATE OR REPLACE SECRET fabcon_private_flights (
            TYPE azure,
            CONNECTION_STRING
                'BlobEndpoint=https://YOUR-STORAGE-ACCOUNT.blob.core.windows.net/;SharedAccessSignature={sas_token}'
        )
    """)

    # Query the private Parquet file and display the result.
    con.sql(f"""
        SELECT count(*) AS flights_from_barcelona
        FROM read_parquet('{PRIVATE_BLOB}')
        WHERE adep = 'LEBL'
    """).show()

else:
    print("Private Blob example skipped. Set RUN_PRIVATE_BLOB = True if you have access.")


## Optional: an Excel workbook in the Lakehouse
**Same extension pattern, another reader.** The `excel` extension adds `read_xlsx`.
<br>`0. Setup` section 7 writes January's Barcelona departures by destination to a workbook in **Files**.
<br>The workbook holds counts by destination. Sum its `flights` column to recover the total.
<br>Excel stores numbers as doubles, so the cell casts the total back to a whole number.
<br>`read_xlsx` reads `.xlsx` files, not older `.xls` files.
Set `RUN_EXCEL = True` after running `0. Setup` section 7. Leave it `False` to skip.


In [ ]:
RUN_EXCEL = False

if RUN_EXCEL:
    from pathlib import Path

    EXCEL = "/lakehouse/default/Files/flights_eu/excel/barcelona_departures_202501.xlsx"

    if Path(EXCEL).exists():
        con.execute("INSTALL excel")
        con.execute("LOAD excel")

        con.sql(f"""
            SELECT CAST(sum(flights) AS BIGINT) AS flights_from_barcelona
            FROM read_xlsx('{EXCEL}')
        """).show()
    else:
        print("Workbook not found. Run 0. Setup section 7 to create it.")
else:
    print("Excel example skipped. Set RUN_EXCEL = True to run it.")


## What changed?
The SQL question stayed the same. The source address and reader changed.
| Source | DuckDB reader |
| --- | --- |
| Lakehouse Parquet file | `read_parquet('/lakehouse/...')` |
| Lakehouse Delta table | `delta_scan('/lakehouse/.../Tables/...')` |
| Public Azure Blob | `read_parquet('https://...')` through `httpfs` |
| OneLake shortcut to Blob | `read_parquet('/lakehouse/.../Files/blob_flights/...')` |
| Private Azure Blob | `read_parquet('azure://...')` with the `azure` extension and Key Vault credential |
| Excel workbook in the Lakehouse | `read_xlsx('/lakehouse/.../Files/...xlsx')` with the `excel` extension |
**Explore data where it lives.**
<br>Create a Lakehouse copy when the workflow needs one.
